# Mangalate: tradução interativa de capítulos
Este notebook NÃO abre túnel, servidor público ou laço para manter GPU ligada. Use arquivos próprios/autorizados e acompanhe a execução. [Regras e limites do Colab](https://research.google.com/colaboratory/intl/pt-BR/faq.html).
1. Ambiente de execução > Alterar tipo > GPU, se disponível. GPU não é garantida.
2. Execute as células em ordem. Básico usa tradução NLLB local (uso pessoal não comercial; confira a licença do modelo). BYOK é opcional: sua API pode cobrar.
3. Envie UM capítulo ZIP/CBZ com imagens. Baixe o CBZ e o pacote de revisão. Não há promessa de acabamento humano automático.
4. Para leitura ao vivo pelo aplicativo, use Docker no PC ou um servidor que você contrate, não este notebook.


In [ ]:
# Instalar versões fixadas; falhas param a célula, não exibem um falso 'Pronto'.
import pathlib, subprocess, sys
UPSTREAM_REV = 'd6de6ab62a7be6afc0d07a3184149cdefbcd32f2'
CRF_REV = '3ca3d8f0cbd9701f4a765f97fbad43f9b2fe80a6'
if sys.version_info < (3, 12): raise RuntimeError('Esta revisão requer Python 3.12 ou superior. Use um runtime atual ou Docker.')
def command(*args, cwd=None): subprocess.run(args, cwd=cwd, check=True)
content = pathlib.Path('/content')
upstream = content / 'manga-image-translator'
if not upstream.exists(): command('git', 'clone', 'https://github.com/zyddnys/manga-image-translator.git', str(upstream))
command('git', 'fetch', 'origin', UPSTREAM_REV, cwd=upstream)
command('git', 'checkout', '--detach', UPSTREAM_REV, cwd=upstream)
requirements = (upstream / 'requirements.txt').read_text().replace('@fix/python313-cython3', '@' + CRF_REV)
(content / 'mangalate-requirements.txt').write_text(requirements)
command('apt-get', 'update', '-qq')
command('apt-get', 'install', '-y', '-qq', 'g++', 'libgl1', 'libvulkan1', 'fonts-dejavu-core')
command(sys.executable, '-m', 'pip', 'install', '-q', '-r', str(content / 'mangalate-requirements.txt'))
project = content / 'Mangalate'
if not project.exists(): command('git', 'clone', '--depth', '1', 'https://github.com/delkrusty/Mangalate.git', str(project))
command(sys.executable, '-m', 'pip', 'check')
print('Instalação terminou. Se o Colab pedir reinicialização, reinicie e reexecute esta célula antes de continuar.')


In [ ]:
# Configuração: nenhuma chave de API aparece no notebook ou nos resultados.
import os, getpass
PROFILE = input('Perfil: 1 Básico / 2 BYOK (1): ').strip()
PROFILE = 'byok' if PROFILE == '2' else 'basic'
if PROFILE == 'byok':
    print('Você paga sua API diretamente ao provedor. Mangalate não cobra nem recebe comissão.')
    os.environ['CUSTOM_OPENAI_API_BASE'] = input('Endpoint HTTPS Chat Completions do seu provedor: ').strip()
    os.environ['CUSTOM_OPENAI_MODEL'] = input('Nome exato do modelo: ').strip()
    from google.colab import userdata
    try: key = userdata.get('MANGALATE_PROVIDER_KEY')
    except Exception: key = getpass.getpass('Chave API (oculta): ')
    if not key or not os.environ['CUSTOM_OPENAI_API_BASE'].startswith('https://'): raise ValueError('BYOK requer HTTPS e chave.')
    os.environ['CUSTOM_OPENAI_API_KEY'] = key
    del key
import torch
USE_GPU = torch.cuda.is_available()
print('GPU disponível:', USE_GPU, '| Perfil:', PROFILE)
if not USE_GPU: print('CPU pode ser lenta. Você pode interromper e selecionar GPU, se oferecida pelo Colab.')


In [ ]:
# Enviar UM capítulo ZIP/CBZ (até 300 páginas e 500 MB descompactados).
from google.colab import files
uploaded = files.upload()
archives = [pathlib.Path(name) for name in uploaded if pathlib.Path(name).suffix.lower() in {'.zip', '.cbz'}]
if len(archives) != 1: raise ValueError('Envie exatamente um capítulo ZIP ou CBZ com imagens.')
INPUT = archives[0].resolve()
del uploaded


In [ ]:
# Traduzir sequencialmente e gerar pacote de revisão. Não feche esta célula.
OUTPUT = pathlib.Path('/content/mangalate-review')
if OUTPUT.exists(): raise RuntimeError('Saída já existe: baixe o resultado antes de apagar/renomear a pasta para outro capítulo.')
script = pathlib.Path('/content/Mangalate/server/colab/batch.py')
env = os.environ.copy()
env['PYTHONPATH'] = str(upstream) + os.pathsep + env.get('PYTHONPATH', '')
cmd = [sys.executable, str(script), '--input', str(INPUT), '--output', str(OUTPUT), '--profile', PROFILE, '--target', 'PTB']
if USE_GPU: cmd.append('--gpu')
subprocess.run(cmd, cwd=upstream, env=env, check=True)
files.download(str(OUTPUT / 'capitulo-para-revisar.cbz'))
files.download(str(OUTPUT / 'revisao-completa.zip'))
print('Revise arte, balões, nomes, ordem e texto. Páginas bloqueadas preservam o original e aparecem no review.json.')
